# PyTorch DeepDream

This notebook is based on the supplied `test_pytorch.py` implementation.

It demonstrates:
- pretrained InceptionV3
- intermediate-layer activation hooks
- gradient ascent on the input image
- tiled gradient computation
- multi-scale octave processing
- comparison between naive and multi-scale DeepDream

Unlike the original command-line script, this version exposes the parameters directly in notebook cells and displays intermediate results.

## Install dependencies

Run this cell if the environment does not already contain the required packages.

In [ ]:
# Uncomment if needed:
# %pip install torch torchvision numpy pillow scipy matplotlib

## Imports and device

In [ ]:
import os
import numpy as np
import PIL.Image
import scipy.ndimage
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

## Load pretrained InceptionV3

DeepDream works by maximizing the activation of an intermediate neural-network layer with respect to the input image.

The network weights remain frozen; only the image is modified.

In [ ]:
model = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
model = model.to(device)
model.eval()

for param in model.parameters():
    param.requires_grad = False

imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
imagenet_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

target_layers = [
    'Mixed_5b', 'Mixed_5c', 'Mixed_6a', 'Mixed_6b',
    'Mixed_6c', 'Mixed_7a', 'Mixed_7b', 'Mixed_7c'
]

print('InceptionV3 loaded.')

## Register activation hooks

Forward hooks let us capture intermediate feature maps during a forward pass.

In [ ]:
layer_outputs = {}


def create_hook(layer_name):
    """Create a forward hook to capture layer output."""
    def hook(module, input, output):
        layer_outputs[layer_name] = output
    return hook


hooks = []
for name, module in model.named_modules():
    if any(target in name for target in target_layers):
        hook = module.register_forward_hook(create_hook(name))
        hooks.append(hook)

print(f'Registered {len(hooks)} hooks.')

## Image utilities

The model expects ImageNet-normalized tensors with shape `[1, 3, H, W]`.

In [ ]:
img_noise = np.random.uniform(size=(224, 224, 3)) + 100.0


def showarray(a, fname='out_pytorch.jpg'):
    """Save a numpy image array in the [0, 1] range."""
    a = np.uint8(np.clip(a, 0, 1) * 255)
    img = PIL.Image.fromarray(a)
    img.save(fname)
    print(f'Saved image to {fname}')


def visstd(a, s=0.1):
    """Normalize an image for visualization."""
    return (a - a.mean()) / max(a.std(), 1e-4) * s + 0.5


def img_to_tensor(img_np):
    """Convert [H, W, 3] numpy image to normalized [1, 3, H, W] tensor."""
    img_tensor = torch.from_numpy(img_np).float().to(device)
    img_tensor = img_tensor / 255.0
    img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0)
    img_tensor = (img_tensor - imagenet_mean) / imagenet_std
    return img_tensor


def tensor_to_img(img_tensor):
    """Convert [1, 3, H, W] tensor to [H, W, 3] numpy array."""
    return img_tensor.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()


def resize(img, size):
    """Resize an image using scipy interpolation."""
    factors = [1.0 * size[0] / img.shape[0], 1.0 * size[1] / img.shape[1]]
    factors += [1.0] * (len(img.shape) - 2)
    return scipy.ndimage.zoom(img, factors, order=1)

## Load or generate a starting image

Set `CONTENT_PATH` to an image if you want to apply DeepDream to an existing photograph. If the file is missing, the notebook falls back to random noise.

In [ ]:
CONTENT_PATH = 'source.jpg'

if os.path.exists(CONTENT_PATH):
    print(f'Loading content image from: {CONTENT_PATH}')
    img0 = np.float32(PIL.Image.open(CONTENT_PATH).convert('RGB'))
else:
    print(f'Content image not found: {CONTENT_PATH}')
    print('Generating from random noise instead.')
    img0 = img_noise.copy()

print(f'Input image shape: {img0.shape}')

plt.figure(figsize=(7, 7))
plt.imshow(np.clip(img0 / 255.0, 0, 1))
plt.title('Starting image')
plt.axis('off')
plt.show()

## Extract a target layer activation

In [ ]:
def compute_layer_activation(img_tensor, layer_name):
    """Forward pass and retrieve a selected intermediate activation."""
    layer_outputs.clear()
    model(img_tensor)

    if layer_name in layer_outputs:
        return layer_outputs[layer_name]

    for key in layer_outputs.keys():
        if layer_name in key:
            return layer_outputs[key]

    raise ValueError(f'Layer {layer_name} not found in model')

## Tiled gradient computation

Large images can consume substantial memory. The tiled implementation splits the image into smaller regions and calculates gradients separately.

A random spatial shift is applied before tiling and undone afterward to reduce visible tile boundaries.

In [ ]:
def calc_grad_tiled(img, layer_name, tile_size=512):
    """Compute activation gradients with respect to an image using tiles."""
    sz = tile_size
    h, w = img.shape[:2]

    sx, sy = np.random.randint(sz, size=2)
    img_shift = np.roll(np.roll(img, sx, 1), sy, 0)
    grad = np.zeros_like(img)

    for y in range(0, max(h - sz // 2, sz), sz):
        for x in range(0, max(w - sz // 2, sz), sz):
            sub = img_shift[y:y + sz, x:x + sz]

            img_tensor = img_to_tensor(sub)
            img_tensor.requires_grad_(True)

            layer_output = compute_layer_activation(img_tensor, layer_name)
            loss = layer_output.mean()
            loss.backward()

            g = img_tensor.grad.data
            g = tensor_to_img(g)
            grad[y:y + sz, x:x + sz] = g

    return np.roll(np.roll(grad, -sx, 1), -sy, 0)

## Naive DeepDream

This is the simplest version: repeatedly maximize the mean activation of one layer using gradient ascent.

In [ ]:
def render_naive(layer_name, img0, iter_n=20, step=1.0, output='deepdream_naive_pytorch.jpg'):
    """Simple DeepDream without multi-scale processing."""
    img = img0.copy()

    for i in range(iter_n):
        img_tensor = img_to_tensor(img)
        img_tensor.requires_grad_(True)

        layer_output = compute_layer_activation(img_tensor, layer_name)
        loss = layer_output.mean()
        loss.backward()

        g = img_tensor.grad.data
        g = tensor_to_img(g)
        g /= g.std() + 1e-8
        img += g * step

        if i % 5 == 0:
            print(f'Iteration {i + 1}/{iter_n}, Loss: {loss.item():.6f}')

    showarray(visstd(img), output)
    return img

## Multi-scale DeepDream

The octave approach processes the image at several scales. Coarse structures are enhanced first, then high-frequency details from finer resolutions are reintroduced.

In [ ]:
def render_deepdream(
    layer_name,
    img0,
    iter_n=15,
    step=0.5,
    octave_n=3,
    octave_scale=1.3,
    output='deepdream_pytorch.jpg',
):
    """Generate a DeepDream image using multi-scale octave processing."""
    img = img0.copy()
    octaves = []

    for i in range(octave_n - 1):
        hw = img.shape[:2]
        lo = resize(img, np.int32(np.float32(hw) / octave_scale))
        hi = img - resize(lo, hw)
        img = lo
        octaves.append(hi)

    for octave in range(octave_n):
        if octave > 0:
            hi = octaves[-octave]
            img = resize(img, hi.shape[:2]) + hi

        print(f'Processing octave {octave + 1}/{octave_n}')

        for i in range(iter_n):
            g = calc_grad_tiled(img, layer_name, tile_size=256)
            g /= g.std() + 1e-8
            img += g * step

            if i % 5 == 0:
                print(f'  Iteration {i + 1}/{iter_n}')

    showarray(img / 255.0, output)
    return img

## Run DeepDream

These values correspond to the defaults in the supplied script.

In [ ]:
LAYER = 'Mixed_5c'
ITERATIONS = 15
STEP = 0.5
OCTAVES = 3
OUTPUT = 'deepdream_pytorch.jpg'

if LAYER not in target_layers:
    raise ValueError(f'LAYER must be one of: {target_layers}')

dream_image = render_deepdream(
    LAYER,
    img0,
    iter_n=ITERATIONS,
    step=STEP,
    octave_n=OCTAVES,
    octave_scale=1.3,
    output=OUTPUT,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(np.clip(img0 / 255.0, 0, 1))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(np.clip(dream_image / 255.0, 0, 1))
axes[1].set_title(f'DeepDream: {LAYER}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Experiment with different layers

Different InceptionV3 layers tend to produce different visual characteristics. Try the following one at a time:

```text
Mixed_5b
Mixed_5c
Mixed_6a
Mixed_6b
Mixed_6c
Mixed_7a
Mixed_7b
Mixed_7c
```

Earlier layers generally emphasize lower-level patterns, while deeper layers tend to produce more complex semantic-looking structures.

## Cleanup

Remove the hooks when you are finished with the model.

In [ ]:
for hook in hooks:
    hook.remove()

print('Hooks removed.')